# QLNH — Gợi ý món ăn bằng Apriori

Notebook chạy độc lập trên Google Colab, không cần kết nối database hoặc clone repository.
**Mặc định: dữ liệu mô phỏng `synthetic_demo`.** Các cặp món trong demo được tạo chủ ý;
kết quả chỉ chứng minh quy trình chạy được, không chứng minh độ chính xác kinh doanh thực tế.

Với CSV xuất từ ứng dụng, đổi `DATA_MODE` thành `csv` ở ô cấu hình. Nguồn
`application_export` không tự khẳng định đó là dữ liệu nhà hàng thật.

Quy trình: kiểm tra CSV → gom giỏ → chia theo ngày kinh doanh → Apriori trên train →
kiểm tra trên holdout mới hơn → so sánh món phổ biến → lưu luật và báo cáo JSON/CSV.

In [ ]:
%pip install -q numpy==2.2.6 pandas==2.3.3 scikit-learn==1.7.2 mlxtend==0.23.4

## Mã xử lý

Bốn ô dưới đây được tạo từ `ai_tools/schema.py`, `inference.py`, `demo.py`, `apriori.py` và được kiểm thử
cùng mã CLI. Chạy lần lượt; thông thường bạn chỉ cần sửa ô cấu hình bên dưới.

In [ ]:
SCHEMA_VERSION = 1
TRANSACTION_FIELDS = (
    "schema_version", "data_source", "basket_id", "basket_closed_at",
    "dish_id", "dish_code", "dish_name", "quantity",
)
DATA_SOURCES = {"application_export", "synthetic_demo"}

In [ ]:
"""Validated Apriori inference using only the Python standard library."""

import json
import math
from pathlib import Path


class InvalidAprioriModel(ValueError):
    pass


def validate_model(payload):
    if not isinstance(payload, dict) or type(payload.get("schema_version")) is not int or payload["schema_version"] != 1:
        raise InvalidAprioriModel("Unsupported model schema")
    if payload.get("algorithm") != "Apriori" or payload.get("data_source") not in {"application_export", "synthetic_demo"}:
        raise InvalidAprioriModel("Unsupported model algorithm or source")
    catalog, rules = payload.get("catalog"), payload.get("rules")
    if not isinstance(catalog, list) or not 1 <= len(catalog) <= 5000 or not isinstance(rules, list) or len(rules) > 5000:
        raise InvalidAprioriModel("Invalid model size")
    identifiers, codes = set(), set()
    for item in catalog:
        if not isinstance(item, dict) or any(not isinstance(item.get(key), str) or not item[key].strip() or len(item[key]) > 200
                                             for key in ("dish_id", "dish_code", "dish_name")):
            raise InvalidAprioriModel("Invalid catalog item")
        if item["dish_id"] in identifiers or item["dish_code"] in codes:
            raise InvalidAprioriModel("Duplicate catalog identifier")
        identifiers.add(item["dish_id"])
        codes.add(item["dish_code"])
    normalized = []
    for rule in rules:
        if not isinstance(rule, dict):
            raise InvalidAprioriModel("Invalid rule")
        left, right = rule.get("antecedents"), rule.get("consequents")
        if any(not isinstance(values, list) or not 1 <= len(values) <= 10 or
               any(not isinstance(value, str) or value not in identifiers for value in values)
               for values in (left, right)):
            raise InvalidAprioriModel("Invalid rule items")
        if len(set(left)) != len(left) or len(set(right)) != len(right) or set(left) & set(right):
            raise InvalidAprioriModel("Invalid rule overlap")
        metrics = {}
        for key in ("support", "confidence", "lift"):
            value = rule.get(key)
            if type(value) not in (int, float) or not math.isfinite(value) or value < 0 or (key != "lift" and value > 1):
                raise InvalidAprioriModel("Invalid rule metrics")
            metrics[key] = float(value)
        normalized.append({"antecedents": sorted(left), "consequents": sorted(right), **metrics})
    normalized.sort(key=lambda rule: (-rule["confidence"], -rule["lift"], -rule["support"], rule["antecedents"], rule["consequents"]))
    return {**payload, "rules": normalized}


def load_model(path):
    path = Path(path)
    if path.stat().st_size > 2_000_000:
        raise InvalidAprioriModel("Model exceeds size limit")
    try:
        payload = json.loads(path.read_text(encoding="utf-8-sig"))
    except (ValueError, UnicodeError, RecursionError) as error:
        raise InvalidAprioriModel("Invalid model JSON") from error
    return validate_model(payload)


def recommend(cart, rules, *, k=3, allowed_items=None):
    if k < 1:
        raise ValueError("k phải dương.")
    selected, suggestions = set(cart), []
    allowed = set(allowed_items) if allowed_items is not None else None
    for rule in rules:
        if set(rule["antecedents"]) <= selected:
            for item in rule["consequents"]:
                if item not in selected and item not in suggestions and (allowed is None or item in allowed):
                    suggestions.append(item)
                    if len(suggestions) == k:
                        return suggestions
    return suggestions

In [ ]:
"""Reproducible fictional baskets; never connects to Django or its database."""

import csv
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path
import random



DEMO_DISHES = (
    ("DEMO01", "Cơm gà"), ("DEMO02", "Canh rau"), ("DEMO03", "Trà đá"),
    ("DEMO04", "Bún bò"), ("DEMO05", "Nem rán"), ("DEMO06", "Nước cam"),
    ("DEMO07", "Cá kho"), ("DEMO08", "Cơm trắng"), ("DEMO09", "Rau xào"),
    ("DEMO10", "Lẩu nấm"), ("DEMO11", "Mì"), ("DEMO12", "Chè"),
)


def demo_rows(*, baskets=2000, days=180, seed=42, start=date(2026, 1, 1)):
    if baskets < 2 or days < 2:
        raise ValueError("Cần ít nhất 2 giỏ món và 2 ngày để chia tập theo thời gian.")
    rng = random.Random(seed)
    names = dict(DEMO_DISHES)
    bundles = [codes for codes in (
        ("DEMO01", "DEMO02", "DEMO03"), ("DEMO04", "DEMO05", "DEMO06"),
        ("DEMO07", "DEMO08", "DEMO09"), ("DEMO10", "DEMO11", "DEMO03"),
    )]
    for index in range(baskets):
        day = start + timedelta(days=index * days // baskets)
        closed_at = datetime.combine(day, time(rng.choice((11, 12, 18, 19, 20)), rng.randrange(60)),
                                     tzinfo=timezone(timedelta(hours=7)))
        # These associations are deliberately planted for a reproducible demo.
        bundle = rng.choice(bundles)
        chosen = {bundle[0]}
        chosen.update(code for code in bundle[1:] if rng.random() < 0.8)
        if rng.random() < 0.3:
            chosen.add(rng.choice(DEMO_DISHES)[0])
        if rng.random() < 0.2:
            chosen.add("DEMO12")
        for code in sorted(chosen):
            yield {
                "schema_version": SCHEMA_VERSION, "data_source": "synthetic_demo",
                "basket_id": f"DEMO-{index + 1:06d}", "basket_closed_at": closed_at.isoformat(),
                "dish_id": code, "dish_code": code, "dish_name": names[code],
                "quantity": rng.choice((1, 1, 1, 2, 3)),
            }


def write_demo_csv(path, **options):
    rows = iter(demo_rows(**options))
    first = next(rows)  # Validate options before creating any file.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=TRANSACTION_FIELDS)
        writer.writeheader()
        writer.writerow(first)
        writer.writerows(rows)
    return path

In [ ]:
"""Apriori experiment shared by the CLI and the self-contained Colab notebook."""

from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import mlxtend
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder



def load_transactions(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    missing = set(TRANSACTION_FIELDS) - set(frame.columns)
    if missing:
        raise ValueError(f"CSV thiếu các cột: {', '.join(sorted(missing))}")
    frame = frame[list(TRANSACTION_FIELDS)].copy()
    if frame.empty:
        raise ValueError("CSV chưa có giỏ món hợp lệ. Xuất thêm dữ liệu hoặc chạy chế độ demo.")
    for field in TRANSACTION_FIELDS:
        frame[field] = frame[field].str.strip()
        if frame[field].eq("").any():
            raise ValueError(f"Cột {field} có giá trị trống.")
    if set(frame["schema_version"]) != {str(SCHEMA_VERSION)}:
        raise ValueError("Phiên bản schema CSV chưa được hỗ trợ.")
    sources = set(frame["data_source"])
    if len(sources) != 1 or not sources <= DATA_SOURCES:
        raise ValueError("Chỉ dùng một nguồn application_export hoặc synthetic_demo trong mỗi lần chạy.")
    if not frame["basket_closed_at"].str.contains(r"(?:Z|[+-]\d{2}:\d{2})$", regex=True).all():
        raise ValueError("basket_closed_at phải có múi giờ, ví dụ 2026-01-01T12:00:00+07:00.")
    frame["basket_closed_at"] = pd.to_datetime(frame["basket_closed_at"], format="ISO8601", utc=True)
    quantity = pd.to_numeric(frame["quantity"], errors="raise")
    if not (np.isfinite(quantity) & quantity.gt(0) & quantity.mod(1).eq(0)).all():
        raise ValueError("quantity phải là số nguyên dương hữu hạn.")
    frame["quantity"] = quantity
    if frame.groupby("basket_id")["basket_closed_at"].nunique().gt(1).any():
        raise ValueError("Một basket_id phải có cùng thời điểm đóng giỏ ở mọi dòng.")
    for field in ("dish_code", "dish_name"):
        if frame.groupby("dish_id")[field].nunique().gt(1).any():
            raise ValueError(f"Một dish_id có nhiều giá trị {field}; cần chuẩn hóa lại danh mục.")
    return frame.drop_duplicates().reset_index(drop=True)


def split_baskets(frame, *, test_fraction=0.2):
    if not 0 < test_fraction < 1:
        raise ValueError("test_fraction phải nằm giữa 0 và 1.")
    baskets = frame.groupby("basket_id", sort=True).agg(
        closed_at=("basket_closed_at", "first"),
        items=("dish_id", lambda values: tuple(sorted(set(values)))),
    ).reset_index().sort_values(["closed_at", "basket_id"])
    baskets["business_day"] = baskets["closed_at"].dt.tz_convert("Asia/Ho_Chi_Minh").dt.date
    days = sorted(baskets["business_day"].unique())
    if len(days) < 2:
        raise ValueError("Cần ít nhất 2 ngày có giỏ món để tách tập kiểm tra theo thời gian.")
    test_days = min(len(days) - 1, max(1, math.ceil(len(days) * test_fraction)))
    cutoff = days[-test_days]
    train = baskets[baskets["business_day"] < cutoff].copy()
    test = baskets[baskets["business_day"] >= cutoff].copy()
    return train, test, cutoff


def fit_rules(baskets, *, min_support=0.04, min_confidence=0.3, min_lift=1.05, max_len=3, max_rules=500):
    if not (0 < min_support <= 1 and 0 < min_confidence <= 1 and math.isfinite(min_lift) and min_lift > 0):
        raise ValueError("Ngưỡng support/confidence phải thuộc (0, 1], lift phải dương hữu hạn.")
    if max_len < 2 or max_rules < 1:
        raise ValueError("max_len phải từ 2 và max_rules phải dương.")
    transactions = baskets["items"].tolist()
    if not transactions:
        raise ValueError("Tập huấn luyện trống.")
    encoder = TransactionEncoder()
    matrix = encoder.fit(transactions).transform(transactions)
    encoded = pd.DataFrame(matrix, columns=encoder.columns_, dtype=bool)
    frequent = apriori(encoded, min_support=min_support, use_colnames=True, max_len=max_len)
    if frequent.empty or not frequent["itemsets"].map(len).ge(2).any():
        return [], frequent
    discovered = association_rules(frequent, num_itemsets=len(encoded), metric="confidence",
                                   min_threshold=min_confidence, return_metrics=["support", "confidence", "lift"])
    discovered = discovered[discovered["consequents"].map(len).eq(1) & discovered["lift"].ge(min_lift)]
    rules = [{
        "antecedents": sorted(row.antecedents), "consequents": sorted(row.consequents),
        "support": float(row.support), "confidence": float(row.confidence), "lift": float(row.lift),
    } for row in discovered.itertuples()]
    rules.sort(key=lambda rule: (-rule["confidence"], -rule["lift"], -rule["support"], rule["antecedents"], rule["consequents"]))
    return rules[:max_rules], frequent


def evaluate_rules(train, test, rules, *, k=3):
    popularity = Counter(item for items in train["items"] for item in set(items))
    ranked_popular = sorted(popularity, key=lambda item: (-popularity[item], item))
    eligible = hits = baseline_hits = covered = 0
    for basket in test.itertuples():
        if len(basket.items) < 2:
            continue
        hidden_index = int.from_bytes(hashlib.sha256(basket.basket_id.encode()).digest()[:8], "big") % len(basket.items)
        hidden = basket.items[hidden_index]
        observed = set(basket.items) - {hidden}
        predictions = recommend(observed, rules, k=k)
        baseline = [item for item in ranked_popular if item not in observed][:k]
        eligible += 1
        covered += bool(predictions)
        hits += hidden in predictions
        baseline_hits += hidden in baseline
    return {
        "protocol": "chronological_business_days_then_hide_one_item_sha256_basket_id",
        "k": k, "test_baskets": len(test), "eligible_baskets": eligible,
        "single_item_baskets_skipped": len(test) - eligible,
        "hit_rate_at_k": hits / eligible if eligible else None,
        "recommendation_coverage": covered / eligible if eligible else None,
        "popularity_baseline_hit_rate_at_k": baseline_hits / eligible if eligible else None,
        "note": "Giấu một món trong giỏ hoàn tất để kiểm tra; không phải đánh giá thứ tự chọn món thực tế.",
    }


def run_experiment(csv_path, output_dir, *, min_support=0.04, min_confidence=0.3, min_lift=1.05,
                   max_len=3, max_rules=500, test_fraction=0.2, k=3):
    frame = load_transactions(csv_path)
    train, test, cutoff = split_baskets(frame, test_fraction=test_fraction)
    parameters = {"min_support": min_support, "min_confidence": min_confidence,
                  "min_lift": min_lift, "max_len": max_len, "max_rules": max_rules}
    rules, frequent = fit_rules(train, **parameters)
    metrics = evaluate_rules(train, test, rules, k=k)
    source = frame["data_source"].iloc[0]
    catalog = frame[["dish_id", "dish_code", "dish_name"]].drop_duplicates("dish_id").sort_values("dish_id")
    model = {
        "schema_version": SCHEMA_VERSION, "algorithm": "Apriori", "data_source": source,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "versions": {"mlxtend": mlxtend.__version__, "pandas": pd.__version__, "numpy": np.__version__},
        "input_sha256": hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),
        "parameters": {**parameters, "test_fraction": test_fraction, "k": k},
        "training": {"baskets": len(train), "holdout_baskets": len(test),
                     "holdout_start_business_day": cutoff.isoformat(), "rule_count": len(rules),
                     "train_start": train["closed_at"].min().isoformat(), "train_end": train["closed_at"].max().isoformat()},
        "limitations": [
            "Dữ liệu synthetic_demo có các cặp món được tạo chủ ý, chỉ dùng kiểm tra quy trình.",
            "application_export xác nhận nguồn ứng dụng, không xác nhận đây là dữ liệu kinh doanh thực.",
            "Support/confidence/lift đo liên hệ trong tập train, không phải độ chính xác ngoài thực tế.",
            "Chọn ngưỡng trước khi xem holdout; thử nhiều ngưỡng trên holdout làm kết quả mang tính thăm dò.",
            "Khi tích hợp web phải lọc lại món còn bán bằng trạng thái hiện tại của hệ thống.",
        ],
        "catalog": catalog.to_dict("records"), "evaluation": metrics, "rules": rules,
    }
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=False)
    for name, payload in (("apriori_model.json", model), ("evaluation.json", metrics)):
        (output / name).write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    rule_frame = pd.DataFrame(rules, columns=["antecedents", "consequents", "support", "confidence", "lift"])
    for column in ("antecedents", "consequents"):
        rule_frame[column] = rule_frame[column].map(lambda items: json.dumps(items, ensure_ascii=False))
    rule_frame.to_csv(output / "association_rules.csv", index=False, encoding="utf-8-sig")
    frequent = frequent.copy()
    frequent["itemsets"] = frequent["itemsets"].map(lambda items: json.dumps(sorted(items), ensure_ascii=False))
    frequent.to_csv(output / "frequent_itemsets.csv", index=False, encoding="utf-8-sig")
    for name, fold in (("train_baskets.csv", train), ("holdout_baskets.csv", test)):
        fold = fold.copy()
        fold["items"] = fold["items"].map(lambda items: json.dumps(items, ensure_ascii=False))
        fold.to_csv(output / name, index=False, encoding="utf-8-sig")
    return model

## Cấu hình và chọn dữ liệu

- `DATA_MODE = "demo"`: tạo 2.000 giỏ giả lập trong 180 ngày, không ghi database.
- `DATA_MODE = "csv"`: dùng CSV từ `manage.py export_ai_transactions`. Trên Colab sẽ
  hiện hộp chọn file nếu `CSV_PATH` chưa tồn tại. File không chứa tên/SĐT khách.
- Chọn ngưỡng trước khi xem kết quả holdout. Không trộn dữ liệu giả lập vào dữ liệu ứng dụng.

In [ ]:
DATA_MODE = "demo"  # "demo" hoặc "csv"
CSV_PATH = Path("transactions.csv")
DEMO_BASKETS = 2000
DEMO_DAYS = 180
SEED = 42
MIN_SUPPORT = 0.04
MIN_CONFIDENCE = 0.30
MIN_LIFT = 1.05
TOP_K = 3
TEST_FRACTION = 0.20
DOWNLOAD_RESULTS = True

In [ ]:
import shutil
try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORK_DIR = Path("/content/qlnh_apriori") if IN_COLAB else Path("artifacts/ai/notebook")
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

if DATA_MODE == "demo":
    INPUT_CSV = write_demo_csv(WORK_DIR / f"synthetic_demo_{RUN_ID}.csv",
                               baskets=DEMO_BASKETS, days=DEMO_DAYS, seed=SEED)
elif DATA_MODE == "csv":
    INPUT_CSV = CSV_PATH
    if not INPUT_CSV.is_file() and IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(csv_names) != 1:
            raise ValueError("Chọn đúng một file CSV giao dịch.")
        INPUT_CSV = Path(csv_names[0])
    if not INPUT_CSV.is_file():
        raise FileNotFoundError("Chưa có CSV. Xuất dữ liệu rồi đặt CSV_PATH đúng vị trí file.")
else:
    raise ValueError("DATA_MODE chỉ nhận demo hoặc csv.")

transactions = load_transactions(INPUT_CSV)
if DATA_MODE == "demo" and transactions["data_source"].iloc[0] != "synthetic_demo":
    raise ValueError("Chế độ demo phải dùng dữ liệu synthetic_demo.")
print("Nguồn:", transactions["data_source"].iloc[0])
print("Số giỏ:", transactions["basket_id"].nunique(), "| Số món:", transactions["dish_id"].nunique())
display(transactions.head())

## Khai phá luật và đánh giá

Một giỏ là một lượt khách, kể cả khi tách thành nhiều hóa đơn. Chỉ lấy món đã phục vụ
trong các lượt đã hoàn tất thanh toán. Số lượng món không nhân bản số giao dịch.

Giữ khoảng 20% **ngày kinh doanh cuối** làm holdout, không chia ngẫu nhiên từng dòng.
Giỏ một món vẫn tham gia khai phá support nhưng không dùng bài kiểm tra giấu một món.
Mỗi giỏ holdout có từ hai món được giấu một món bằng quy tắc SHA-256 cố định của basket_id;
các món còn lại là đầu vào gợi ý. `hit_rate_at_k` là tỷ lệ tìm lại được món giấu trong top K;
`recommendation_coverage` là tỷ lệ giỏ có ít nhất một gợi ý. So sánh với top món phổ biến
chỉ tính trên train. Đây là bài kiểm tra giỏ hàng, không đo thứ tự gọi món thực tế.

In [ ]:
OUTPUT_DIR = WORK_DIR / f"apriori_{RUN_ID}"
MODEL = run_experiment(INPUT_CSV, OUTPUT_DIR, min_support=MIN_SUPPORT,
                       min_confidence=MIN_CONFIDENCE, min_lift=MIN_LIFT,
                       test_fraction=TEST_FRACTION, k=TOP_K)
print(json.dumps({"nguon": MODEL["data_source"], "chia_du_lieu": MODEL["training"],
                  "danh_gia": MODEL["evaluation"]}, ensure_ascii=False, indent=2))
if not MODEL["rules"]:
    print("Không tìm được luật đạt ngưỡng. Đây là kết quả hợp lệ; không tạo luật giả để lấp chỗ trống.")
display(pd.DataFrame(MODEL["rules"]).head(10))

## Thử gợi ý món

Sửa `CART` thành danh sách dish_id của các món đang chọn. Ví dụ mặc định lấy vế trái
của một luật train để minh họa hoạt động, không phải một phép đo chất lượng bổ sung.
Khi tích hợp Django sau này cần lọc lại món còn bán bằng trạng thái thực tế lúc gợi ý.

In [ ]:
CART = list(MODEL["rules"][0]["antecedents"]) if MODEL["rules"] else []
names = {item["dish_id"]: item["dish_name"] for item in MODEL["catalog"]}
suggestions = recommend(CART, MODEL["rules"], k=TOP_K)
print("Đang chọn:", [names.get(item, item) for item in CART])
print("Gợi ý:", [names.get(item, item) for item in suggestions])

## Lưu kết quả

File ZIP gồm `apriori_model.json`, `evaluation.json`, `association_rules.csv`,
`frequent_itemsets.csv`, `train_baskets.csv` và `holdout_baskets.csv`.
JSON lưu luật và metadata, không dùng pickle. Trường `data_source`, hash CSV, phiên bản
thư viện, ngưỡng và ngày chia tập giúp đối chiếu lại thí nghiệm. Nếu không có giỏ holdout
đủ hai món, chỉ số kiểm tra là `null`, không được diễn giải thành 0% hay 100%.

Support/confidence/lift mô tả liên hệ trên train. Dữ liệu ít hoặc dữ liệu mô phỏng cần
được ghi rõ trong báo cáo; không dùng các kết quả đó để tuyên bố độ chính xác thực tế.

In [ ]:
ARCHIVE = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Kết quả:", OUTPUT_DIR.resolve())
print("ZIP:", ARCHIVE)
if IN_COLAB and DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(ARCHIVE)

## Tài liệu tham khảo

- [Apriori — mlxtend](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/)
- [Association rules — mlxtend](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/)

Phạm vi notebook này: đề xuất món bằng Apriori. Phần dự báo số đơn theo ngày là bước tiếp theo.